# 🎓 Campus Lost & Found — Custom YOLOv8 Training Pipeline on Google Colab

This notebook automates:
1. **Environment Setup**: Hardware accelerator verification & package installations (`ultralytics`, `roboflow`).
2. **Roboflow Dataset Ingestion**: Downloads the 6 designated campus datasets:
   - `student-card/student-card-xrkhm/1` (student-card -> ID Card)
   - `samples/keys-jcx6v/1` (keys-F8z5 -> Keys)
   - `senior-project-final-datasets/wallet-nfk4a/2` (Wallet -> Wallet)
   - `breathless/bottle-ewqkq/8` (bottle -> Bottle)
   - `test-djdmx/backpack-enkb9/1` (backpack -> Backpack)
   - `s-workspace-gtpuo/phone-and-laptop/1` (phone, laptop -> Phone, Laptop)
3. **Dataset Merging & Dynamic Remapping**: Merges annotations, drops unmapped classes (`handbag`, `paper bag`), creates dynamic classes, and warns about classes < 100 boxes.
4. **YOLOv8n Fine-Tuning**: Trains for 50 epochs on GPU and calculates per-class `mAP@50`.
5. **Export**: Automatically triggers browser download of the fine-tuned `campus_best.pt` file.

In [ ]:
# Step 1: Verify Hardware Accelerator (GPU)
!nvidia-smi

In [ ]:
# Step 2: Install Required Libraries
!pip install -q ultralytics roboflow pyyaml

### Step 3: Set Up Workspace Directory & Fetch Helper Scripts

In [ ]:
import os

# Ensure local folder structure is ready
os.makedirs("datasets", exist_ok=True)
os.makedirs("../../ai-service/models", exist_ok=True)
print("Workspace directories ready!")

### Step 4: Download the 6 Designated Roboflow Datasets
Enter your Roboflow API key. The script will download all 6 datasets in YOLOv8 format.
If any specific version fails, it will automatically query and download the project's latest available version.

In [ ]:
import os
import getpass
from roboflow import Roboflow

# 1. Input Roboflow API Key
ROBOFLOW_API_KEY = getpass.getpass("Enter your Roboflow API Key: ")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# 2. The 6 Exact Campus Lost & Found Datasets
DATASETS_TO_DOWNLOAD = [
    {"workspace": "student-card", "project": "student-card-xrkhm", "version": 1},
    {"workspace": "samples", "project": "keys-jcx6v", "version": 1},
    {"workspace": "senior-project-final-datasets", "project": "wallet-nfk4a", "version": 2},
    {"workspace": "breathless", "project": "bottle-ewqkq", "version": 8},
    {"workspace": "test-djdmx", "project": "backpack-enkb9", "version": 1},
    {"workspace": "s-workspace-gtpuo", "project": "phone-and-laptop", "version": 1},
]

os.makedirs("datasets", exist_ok=True)

# 3. Download each dataset with automatic fallback to latest version
for item in DATASETS_TO_DOWNLOAD:
    ws = item["workspace"]
    proj_name = item["project"]
    target_version = item["version"]
    save_path = os.path.join("datasets", proj_name)
    
    print(f"\n📥 Fetching [{ws}/{proj_name}] (version {target_version})...")
    try:
        project = rf.workspace(ws).project(proj_name)
        try:
            dataset = project.version(target_version).download("yolov8", location=save_path)
            print(f"   ✅ Downloaded {proj_name} v{target_version}")
        except Exception as ver_err:
            print(f"   ⚠️ Version {target_version} failed ({ver_err}). Falling back to project's latest version...")
            versions_list = project.versions()
            if not versions_list:
                raise RuntimeError(f"No versions found for {proj_name}")
            
            # Extract available version numbers
            ver_ints = []
            for v in versions_list:
                v_val = getattr(v, 'version', None)
                if v_val is not None:
                    try:
                        ver_ints.append(int(v_val))
                    except ValueError:
                        pass
            
            latest_v = max(ver_ints) if ver_ints else versions_list[-1].version
            print(f"   🔄 Downloading latest version v{latest_v} for {proj_name}...")
            dataset = project.version(latest_v).download("yolov8", location=save_path)
            print(f"   ✅ Downloaded {proj_name} v{latest_v} (latest)")
    except Exception as err:
        print(f"   ❌ Failed to download {ws}/{proj_name}: {err}")

print("\nAll available datasets downloaded to 'datasets/' directory.")

### Step 5: Merge Datasets & Dynamically Remap Classes
Runs `merge_datasets.py` to:
- Remap `student-card` -> `ID Card`
- Remap `keys-F8z5` -> `Keys`
- Remap `Wallet` -> `Wallet`
- Remap `bottle` -> `Bottle`
- Remap `backpack` -> `Backpack` (drops `handbag` and `paper bag`)
- Remap `phone`, `laptop` -> `Phone`, `Laptop`
- Build active class list dynamically (skips `Book`, `Umbrella`, `Earphones` if no data without failing)
- Print per-class counts and warn for any class under 100 boxes

In [ ]:
!python merge_datasets.py --datasets-dir datasets --output-dir merged_dataset

### Step 6: Fine-Tune YOLOv8 on GPU
Trains `yolov8n.pt` for 50 epochs on `merged_dataset/data.yaml` and copies `best.pt` to `ai-service/models/campus_best.pt`.

In [ ]:
!python train.py --data merged_dataset/data.yaml --epochs 50 --imgsz 640 --batch 16

### Step 7: Download Trained Weights (`campus_best.pt`)
Downloads the fine-tuned model directly to your computer.

In [ ]:
from google.colab import files
import glob

target_weights = "../../ai-service/models/campus_best.pt"
if not os.path.exists(target_weights):
    runs = glob.glob("runs/**/weights/best.pt", recursive=True)
    if runs:
        target_weights = runs[0]

if os.path.exists(target_weights):
    print(f"Downloading fine-tuned weights: {target_weights}")
    files.download(target_weights)
else:
    print("Trained weights file not found. Check training logs above.")